# HF COLLIDE-1M Quickstart for FlowyForge

This notebook adapts the public Hugging Face COLLIDE-1M quickstart idea to the FlowyForge / M-Fond fast-track data path.

- Dataset: `fastmachinelearning/collide-1m`
- Goal: materialize a small local Parquet subset for inspection and later pploner-style vectorization
- Strategy: keep the pipeline shape as `Parquet -> vectorized .npy -> preprocessing -> training`

The full dataset is large, so this notebook uses Hugging Face streaming by default and caps both rows and output files. It does not require EOS and does not run training.

In [ ]:
from pathlib import Path
import json
import math
from itertools import islice

import pandas as pd

try:
    from datasets import load_dataset
except ImportError:
    load_dataset = None
    print("Missing optional Hugging Face packages. Install with: pip install datasets huggingface_hub")

try:
    from flowyforge.core.config import load_config
    from flowyforge.data_plugins.collide_v2.hf_collide1m import materialize_hf_collide1m_subset
    from flowyforge.data_plugins.collide_v2.source_resolver import resolve_dataset_source
    from flowyforge.data_plugins.collide_v2.schema_inspector import inspect_dataset_schema
    from flowyforge.data_plugins.collide_v2.pploner_adapter import prepare_pploner_paths
except ImportError as exc:
    load_config = None
    materialize_hf_collide1m_subset = None
    resolve_dataset_source = None
    inspect_dataset_schema = None
    prepare_pploner_paths = None
    print(f"FlowyForge imports failed: {exc}")
    print("Install the package from the repository root with: pip install -e .")


In [ ]:
CONFIG_PATH = Path("../configs/paths/hf_collide1m.yaml")
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("configs/paths/hf_collide1m.yaml")

if load_config is None:
    raise RuntimeError("FlowyForge is not importable. Run: pip install -e .")

cfg = load_config(CONFIG_PATH)
paths_cfg = cfg["paths"]
data_cfg = cfg["data"]

dataset_name = paths_cfg["hf_dataset_name"]
split = paths_cfg.get("hf_split", "train")
output_dir = Path(paths_cfg["dataset_dir"])
max_rows = int(data_cfg.get("max_rows") or 10000)
max_files = int(data_cfg.get("max_files") or 2)

print(f"dataset backend: {paths_cfg['dataset_backend']}")
print(f"dataset name: {dataset_name}")
print(f"split: {split}")
print(f"max_rows: {max_rows}")
print(f"max_files: {max_files}")
print(f"output dataset_dir: {output_dir}")


In [ ]:
if load_dataset is None:
    raise RuntimeError("Hugging Face datasets is not installed. Run: pip install datasets huggingface_hub")

preview_rows = list(islice(load_dataset(dataset_name, split=split, streaming=True), 3))
print(f"preview rows: {len(preview_rows)}")

if preview_rows:
    first_row = preview_rows[0]
    print("available keys:", list(first_row.keys()))
    print("first row keys:", list(first_row.keys()))
    lightweight_preview = {}
    for key, value in first_row.items():
        if isinstance(value, (str, int, float, bool)) or value is None:
            lightweight_preview[key] = value
        elif hasattr(value, "shape"):
            lightweight_preview[key] = {"type": type(value).__name__, "shape": tuple(value.shape)}
        elif isinstance(value, (list, tuple)):
            lightweight_preview[key] = {"type": type(value).__name__, "length": len(value)}
        elif isinstance(value, dict):
            lightweight_preview[key] = {"type": "dict", "keys": list(value)[:10]}
        else:
            lightweight_preview[key] = {"type": type(value).__name__}
    print(json.dumps(lightweight_preview, indent=2, default=str))


In [ ]:
if materialize_hf_collide1m_subset is not None:
    manifest = materialize_hf_collide1m_subset(
        dataset_name=dataset_name,
        split=split,
        output_dir=output_dir,
        max_rows=max_rows,
        max_files=max_files,
    )
else:
    if load_dataset is None:
        raise RuntimeError("Install with: pip install datasets huggingface_hub")
    output_dir.mkdir(parents=True, exist_ok=True)
    rows_iter = islice(load_dataset(dataset_name, split=split, streaming=True), max_rows)
    chunk_size = max(1, math.ceil(max_rows / max_files))
    parquet_files = []
    rows_written = 0
    for file_index in range(max_files):
        rows = list(islice(rows_iter, chunk_size))
        if not rows:
            break
        frame = pd.DataFrame(rows)
        parquet_path = output_dir / f"part-{file_index:05d}.parquet"
        frame.to_parquet(parquet_path, index=False)
        parquet_files.append(parquet_path.name)
        rows_written += len(frame)
    manifest = {
        "source_dataset_name": dataset_name,
        "split": split,
        "max_rows": max_rows,
        "max_files": max_files,
        "rows_written": rows_written,
        "parquet_files": parquet_files,
        "warning": "This is a streaming subset, not the full COLLIDE-1M dataset.",
    }
    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    manifest["manifest_path"] = str(manifest_path)

print(json.dumps(manifest, indent=2, default=str))


In [ ]:
if resolve_dataset_source is None or inspect_dataset_schema is None:
    raise RuntimeError("FlowyForge inspection helpers are unavailable. Run: pip install -e .")

source = resolve_dataset_source(cfg)
print(f"resolved backend: {source.backend}")
print(f"parquet files: {len(source.parquet_files)}")
for path in source.parquet_files:
    print(f"- {path}")

if source.parquet_files:
    schema_report = inspect_dataset_schema(source.parquet_files, max_files=max_files)
    print(f"inspected files: {schema_report['num_files_inspected']}")
    print(f"union columns ({len(schema_report['union_columns'])}):")
    for column in schema_report["union_columns"]:
        print(f"- {column}")
else:
    print("No local parquet files found yet. Run the materialization cell first.")


In [ ]:
if prepare_pploner_paths is None:
    raise RuntimeError("FlowyForge pploner adapter is unavailable. Run: pip install -e .")

pipeline_paths = prepare_pploner_paths(source, create_dirs=False)
print(f"dataset_dir: {pipeline_paths.dataset_dir}")
print(f"tmp_data_dir: {pipeline_paths.tmp_data_dir}")
print(f"processed_data_dir: {pipeline_paths.processed_data_dir}")
print(f"file_event_counts_path: {pipeline_paths.file_event_counts_path}")
print(f"split_manifest_path: {pipeline_paths.split_manifest_path}")
print("Vectorization is intentionally not run in this notebook.")


## Next Steps

Use scripts as the reproducible interface once the quick inspection looks right:

```bash
python scripts/prepare_collide_source.py --config configs/paths/hf_collide1m.yaml
python scripts/inspect_dataset.py --config configs/paths/hf_collide1m.yaml
python scripts/vectorize_collide.py --config configs/paths/hf_collide1m.yaml
```

`vectorize_collide.py` currently prepares the backend and pploner-compatible paths; full vectorization is the next implementation step. Full EOS tests happen later on LXPLUS with `configs/paths/eos.yaml`.